In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt


#read in + edit SP 500 value
sp500_df = pd.read_csv("sp500adj.csv")
sp500_df['Date'] = pd.to_datetime(sp500_df['Date'])
sp500_df['date_month'] = sp500_df['Date'].dt.strftime('%Y.%m')
sp500_df = sp500_df.rename(columns={"Value":"SP500 Index Value ($)"})
#sp500_df


#read in + edit geopolitical risk (GPR)
#"Higher geopolitical risk foreshadows lower investment, stock prices, and employment"
gpr_df_raw = pd.read_csv("data_gpr_export.csv")
gpr_df_raw['Date'] = pd.to_datetime(gpr_df_raw['month'])
gpr_df_raw['date_month'] = gpr_df_raw['Date'].dt.strftime('%Y.%m')
us_gpr_df = gpr_df_raw[['date_month','GPRC_USA']]


#read in + edit GDP with inflation adjusted
#note: had to remove first two lines in the dataset in order to actually read it
#those only contained the data source and last edit time:
#"Data Source","World Development Indicators",
#"Last Updated Date","2025-10-07",
GDP_df = pd.read_csv("GDP/API_NY.GDP.MKTP.CD_DS2_en_csv_v2_130122.csv")
GDP_df = GDP_df[GDP_df['Country Code'] == 'USA']
gdp_usa = GDP_df.melt(
    id_vars=['Country Code','Country Name','Indicator Name','Indicator Code'],
    var_name='Year', value_name='GDP'
)
gdp_usa = gdp_usa.rename(columns={"GDP":"US GDP inflation adjusted ($)"})
gdp_usa = gdp_usa[gdp_usa['Year'].str.isnumeric()].copy()
gdp_usa['Year'] = gdp_usa['Year'].astype(int)
#gdp_usa


#read in + edit federal funds rate
fed_df = pd.read_csv("FEDFUNDS.csv")
fed_df['observation_date'] = pd.to_datetime(fed_df['observation_date'])
fed_df['Year'] = fed_df['observation_date'].dt.year
fed_df['FEDFUNDS'] = pd.to_numeric(fed_df['FEDFUNDS'], errors='coerce')
fed_df = fed_df.groupby('Year', as_index=False)['FEDFUNDS'].mean()


#read in + edit CPI (Consumer Price Index)
cpi_df = pd.read_csv("CPIAUCSL.csv")
cpi_df['observation_date'] = pd.to_datetime(cpi_df['observation_date'])
cpi_df['Year'] = cpi_df['observation_date'].dt.year
cpi_df['CPIAUCSL'] = pd.to_numeric(cpi_df['CPIAUCSL'], errors='coerce')
cpi_df = cpi_df.groupby('Year', as_index=False)['CPIAUCSL'].mean()
cpi_df = cpi_df.rename(columns={"CPIAUCSL":"CPI"})


#read in + edit unemployment rate
unrate_df = pd.read_csv("UNRATE.csv")
unrate_df['observation_date'] = pd.to_datetime(unrate_df['observation_date'])
unrate_df['Year'] = unrate_df['observation_date'].dt.year
unrate_df['UNRATE'] = pd.to_numeric(unrate_df['UNRATE'], errors='coerce')
unrate_df = unrate_df.groupby('Year', as_index=False)['UNRATE'].mean()
unrate_df = unrate_df.rename(columns={"UNRATE":"UNEMPLOYMENT RATE"})


#read in + edit VIX (Volatility Index)
vix_df = pd.read_csv("VIXCLS.csv")
vix_df['observation_date'] = pd.to_datetime(vix_df['observation_date'])
vix_df['Year'] = vix_df['observation_date'].dt.year
vix_df['VIXCLS'] = pd.to_numeric(vix_df['VIXCLS'], errors='coerce')
vix_df = vix_df.groupby('Year', as_index=False)['VIXCLS'].mean()
vix_df = vix_df.rename(columns={"VIXCLS":"VIX"})


#merging all the dataframes (by year)
merged_df = pd.merge(sp500_df, us_gpr_df, on='date_month', how='inner').dropna()
merged_df['Year'] = merged_df['Date'].dt.year
annual_df = merged_df.groupby('Year', as_index=False).agg({
    'SP500 Index Value ($)': 'mean',
    'GPRC_USA': 'mean'
})
merged_yearly_df = pd.merge(annual_df, gdp_usa[['Year','US GDP inflation adjusted ($)']], on='Year', how='inner')
merged_yearly_df = pd.merge(merged_yearly_df, fed_df, on='Year', how='left')
merged_yearly_df = pd.merge(merged_yearly_df, cpi_df, on='Year', how='left')
merged_yearly_df = pd.merge(merged_yearly_df, unrate_df, on='Year', how='left')
merged_yearly_df = pd.merge(merged_yearly_df, vix_df, on='Year', how='left')
#merged_yearly_df


#merging all the datasets (by year + month, doesn't include GDP)
merged_df = pd.merge(sp500_df, us_gpr_df, on='date_month', how='inner').dropna()
fed_df = pd.read_csv("FEDFUNDS.csv")
fed_df['observation_date'] = pd.to_datetime(fed_df['observation_date'])
fed_df['date_month'] = fed_df['observation_date'].dt.strftime('%Y.%m')
fed_df = fed_df[['date_month', 'FEDFUNDS']]
merged_monthly_df = pd.merge(merged_df, fed_df, on='date_month', how='left')

#read in + edit CPI for monthly merge
cpi_df_monthly = pd.read_csv("CPIAUCSL.csv")
cpi_df_monthly['observation_date'] = pd.to_datetime(cpi_df_monthly['observation_date'])
cpi_df_monthly['date_month'] = cpi_df_monthly['observation_date'].dt.strftime('%Y.%m')
cpi_df_monthly['CPIAUCSL'] = pd.to_numeric(cpi_df_monthly['CPIAUCSL'], errors='coerce')
cpi_df_monthly = cpi_df_monthly.groupby('date_month', as_index=False)['CPIAUCSL'].mean()
cpi_df_monthly = cpi_df_monthly.rename(columns={"CPIAUCSL":"CPI"})
cpi_df_monthly = cpi_df_monthly[['date_month', 'CPI']]

#read in + edit unemployment rate for monthly merge
unrate_df_monthly = pd.read_csv("UNRATE.csv")
unrate_df_monthly['observation_date'] = pd.to_datetime(unrate_df_monthly['observation_date'])
unrate_df_monthly['date_month'] = unrate_df_monthly['observation_date'].dt.strftime('%Y.%m')
unrate_df_monthly['UNRATE'] = pd.to_numeric(unrate_df_monthly['UNRATE'], errors='coerce')
unrate_df_monthly = unrate_df_monthly.groupby('date_month', as_index=False)['UNRATE'].mean()
unrate_df_monthly = unrate_df_monthly.rename(columns={"UNRATE":"UNEMPLOYMENT RATE"})
unrate_df_monthly = unrate_df_monthly[['date_month', 'UNEMPLOYMENT RATE']]

#read in + edit VIX for monthly merge
vix_df_monthly = pd.read_csv("VIXCLS.csv")
vix_df_monthly['observation_date'] = pd.to_datetime(vix_df_monthly['observation_date'])
vix_df_monthly['date_month'] = vix_df_monthly['observation_date'].dt.strftime('%Y.%m')
vix_df_monthly['VIXCLS'] = pd.to_numeric(vix_df_monthly['VIXCLS'], errors='coerce')
vix_df_monthly = vix_df_monthly.groupby('date_month', as_index=False)['VIXCLS'].mean()
vix_df_monthly = vix_df_monthly.rename(columns={"VIXCLS":"VIX"})
vix_df_monthly = vix_df_monthly[['date_month', 'VIX']]

#merge all monthly datasets
merged_monthly_df = pd.merge(merged_monthly_df, cpi_df_monthly, on='date_month', how='left')
merged_monthly_df = pd.merge(merged_monthly_df, unrate_df_monthly, on='date_month', how='left')
merged_monthly_df = pd.merge(merged_monthly_df, vix_df_monthly, on='date_month', how='left')
merged_monthly_df = merged_monthly_df.sort_values('Date')
merged_monthly_df = merged_monthly_df.set_index('Date')

# Rename columns: CPIAUCSL -> CPI, UNRATE -> UNEMPLOYMENT RATE, VIXCLS -> VIX
merged_monthly_df = merged_monthly_df.rename(columns={
    'CPIAUCSL': 'CPI',
    'UNRATE': 'UNEMPLOYMENT RATE',
    'VIXCLS': 'VIX'
})

# Rename columns in yearly dataset
merged_yearly_df = merged_yearly_df.rename(columns={
    'CPIAUCSL': 'CPI',
    'UNRATE': 'UNEMPLOYMENT RATE',
    'VIXCLS': 'VIX'
})

# To view the datasets, uncomment the following:
# merged_monthly_df
# merged_yearly_df

## data view
You can check Yearly dataset -> merged_yearly_df
You can check Monyhly dataset -> merged_monthly_df

In [ ]:
# Add CHANGE RATE columns for each feature showing percentage change from previous month/year
# For merged_monthly_df and merged_yearly_df

import numpy as np

# Function to add CHANGE RATE columns next to each numeric column
def add_change_rate_columns(df, exclude_cols=None):
    """
    Add CHANGE RATE columns next to each numeric column.
    Each row shows the percentage change from the previous period.
    """
    if exclude_cols is None:
        exclude_cols = []
    
    df_with_changes = df.copy()
    
    # Get numeric columns excluding specified ones
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [col for col in numeric_cols if col not in exclude_cols]
    
    # Create new dataframe with reordered columns
    new_columns = []
    
    for col in df.columns:
        new_columns.append(col)
        # If this is a numeric column, add CHANGE RATE column right after it
        if col in numeric_cols:
            change_col_name = col.replace(' Index Value ($)', ' CHANGE RATE').replace(' ($)', ' CHANGE RATE').replace('_', ' ').title().replace(' ', ' ') + ' CHANGE RATE'
            # Simplify the name
            if 'SP500' in col:
                change_col_name = 'SP500 CHANGE RATE'
            elif 'GPRC_USA' in col or 'GPRC' in col:
                change_col_name = 'GPRC CHANGE RATE'
            elif 'FEDFUNDS' in col or 'FEDFUNDS' in col:
                change_col_name = 'FEDFUNDS CHANGE RATE'
            elif 'CPI' in col and 'CHANGE' not in col:
                change_col_name = 'CPI CHANGE RATE'
            elif 'UNEMPLOYMENT' in col or 'UNRATE' in col:
                change_col_name = 'UNEMPLOYMENT RATE CHANGE RATE'
            elif 'VIX' in col and 'CHANGE' not in col:
                change_col_name = 'VIX CHANGE RATE'
            elif 'GDP' in col:
                change_col_name = 'GDP CHANGE RATE'
            
            # Calculate percentage change
            df_with_changes[change_col_name] = df[col].pct_change(fill_method=None) * 100
            new_columns.append(change_col_name)
    
    # Reorder columns
    df_with_changes = df_with_changes[new_columns]
    
    return df_with_changes

# Apply to merged_monthly_df
merged_monthly_df_with_changes = add_change_rate_columns(merged_monthly_df, exclude_cols=['date_month'])

# Apply to merged_yearly_df
merged_yearly_df_with_changes = add_change_rate_columns(merged_yearly_df, exclude_cols=['Year'])

# To view the datasets, uncomment the following:
#merged_monthly_df_with_changes
# merged_yearly_df_with_changes



In [ ]:
# Normalized graph for monthly dataset
normalized_df = merged_monthly_df.copy()

# Normalize all numeric columns
original_cols = ['SP500 Index Value ($)', 'GPRC_USA', 'FEDFUNDS', 'CPI', 'UNEMPLOYMENT RATE', 'VIX']

# Normalize original columns
for col in original_cols:
    if col in normalized_df.columns:
        normalized_df[col + '_norm'] = (
            normalized_df[col] - normalized_df[col].min()
        ) / (normalized_df[col].max() - normalized_df[col].min())

plt.figure(figsize=(18,8))
# Plot original normalized values
plt.plot(normalized_df.index, normalized_df['SP500 Index Value ($)_norm'], label='S&P 500 (normalized)', linewidth=2)
plt.plot(normalized_df.index, normalized_df['GPRC_USA_norm'], label='Geopolitical Risk (normalized)', linewidth=2)
plt.plot(normalized_df.index, normalized_df['FEDFUNDS_norm'], label='Federal Funds Rate (normalized)', linewidth=2)
if 'CPI_norm' in normalized_df.columns:
    plt.plot(normalized_df.index, normalized_df['CPI_norm'], label='CPI (normalized)', linewidth=2)
if 'UNEMPLOYMENT RATE_norm' in normalized_df.columns:
    plt.plot(normalized_df.index, normalized_df['UNEMPLOYMENT RATE_norm'], label='Unemployment Rate (normalized)', linewidth=2)
if 'VIX_norm' in normalized_df.columns:
    plt.plot(normalized_df.index, normalized_df['VIX_norm'], label='VIX (normalized)', linewidth=2)

plt.title("Normalized Monthly Trends: All Economic Indicators", fontsize=14)
plt.xlabel("Year", fontsize=12)
plt.ylabel("Normalized Value (0–1 scale)", fontsize=12)
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#during spikes in GPR there are sometimes falls in the economy?
#something to investigate might be growth/loss in the SP 500 instead of just total value

## data view (After NaN processing -> If you want to delete NaN in VIX colmun)
You can check Yearly dataset -> merged_yearly_df_clean
You can check Monyhly dataset -> merged_monthly_df_clean

In [ ]:
# Selective Removal - Remove rows with NaN in VIX column
# This approach minimizes data loss while ensuring VIX has complete data

# For Monthly Dataset: Remove rows where VIX is NaN
merged_monthly_df_clean = merged_monthly_df.dropna(subset=['VIX'])

# For Yearly Dataset: Remove rows where VIX is NaN
merged_yearly_df_clean = merged_yearly_df.dropna(subset=['VIX'])

# To view the datasets, uncomment the following:
# merged_monthly_df_clean
# merged_yearly_df_clean

In [ ]:
# Normalized graph for NaN cleaned dataset
normalized_df_clean = merged_monthly_df_clean.copy()

# Normalize all numeric columns
original_cols = ['SP500 Index Value ($)', 'GPRC_USA', 'FEDFUNDS', 'CPI', 'UNEMPLOYMENT RATE', 'VIX']

# Normalize original columns
for col in original_cols:
    if col in normalized_df_clean.columns:
        normalized_df_clean[col + '_norm'] = (
            normalized_df_clean[col] - normalized_df_clean[col].min()
        ) / (normalized_df_clean[col].max() - normalized_df_clean[col].min())

plt.figure(figsize=(18,8))
# Plot original normalized values
plt.plot(normalized_df_clean.index, normalized_df_clean['SP500 Index Value ($)_norm'], label='S&P 500 (normalized)', linewidth=2)
plt.plot(normalized_df_clean.index, normalized_df_clean['GPRC_USA_norm'], label='Geopolitical Risk (normalized)', linewidth=2)
plt.plot(normalized_df_clean.index, normalized_df_clean['FEDFUNDS_norm'], label='Federal Funds Rate (normalized)', linewidth=2)
if 'CPI_norm' in normalized_df_clean.columns:
    plt.plot(normalized_df_clean.index, normalized_df_clean['CPI_norm'], label='CPI (normalized)', linewidth=2)
if 'UNEMPLOYMENT RATE_norm' in normalized_df_clean.columns:
    plt.plot(normalized_df_clean.index, normalized_df_clean['UNEMPLOYMENT RATE_norm'], label='Unemployment Rate (normalized)', linewidth=2)
if 'VIX_norm' in normalized_df_clean.columns:
    plt.plot(normalized_df_clean.index, normalized_df_clean['VIX_norm'], label='VIX (normalized)', linewidth=2)

plt.title("Normalized Monthly Trends (NaN Cleaned): All Economic Indicators", fontsize=14)
plt.xlabel("Year", fontsize=12)
plt.ylabel("Normalized Value (0–1 scale)", fontsize=12)
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Add CHANGE RATE columns for cleaned datasets (merged_monthly_df_clean and merged_yearly_df_clean)
import numpy as np

# Function to add CHANGE RATE columns next to each numeric column
def add_change_rate_columns(df, exclude_cols=None):
    if exclude_cols is None:
        exclude_cols = []
    df_with_changes = df.copy()
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [col for col in numeric_cols if col not in exclude_cols]
    new_columns = []
    for col in df.columns:
        new_columns.append(col)
        if col in numeric_cols:
            if 'SP500' in col:
                change_col_name = 'SP500 CHANGE RATE'
            elif 'GPRC_USA' in col or 'GPRC' in col:
                change_col_name = 'GPRC CHANGE RATE'
            elif 'FEDFUNDS' in col:
                change_col_name = 'FEDFUNDS CHANGE RATE'
            elif 'CPI' in col and 'CHANGE' not in col:
                change_col_name = 'CPI CHANGE RATE'
            elif 'UNEMPLOYMENT' in col or 'UNRATE' in col:
                change_col_name = 'UNEMPLOYMENT RATE CHANGE RATE'
            elif 'VIX' in col and 'CHANGE' not in col:
                change_col_name = 'VIX CHANGE RATE'
            elif 'GDP' in col:
                change_col_name = 'GDP CHANGE RATE'
            else:
                change_col_name = col.replace(' Index Value ($)', ' CHANGE RATE').replace(' ($)', ' CHANGE RATE').replace('_', ' ').title().replace(' ', ' ') + ' CHANGE RATE'
            df_with_changes[change_col_name] = df[col].pct_change(fill_method=None) * 100
            new_columns.append(change_col_name)
    df_with_changes = df_with_changes[new_columns]
    return df_with_changes

# Apply to cleaned datasets
merged_monthly_df_clean_with_changes = add_change_rate_columns(merged_monthly_df_clean, exclude_cols=['date_month'])
merged_yearly_df_clean_with_changes = add_change_rate_columns(merged_yearly_df_clean, exclude_cols=['Year'])

# To view the datasets, uncomment the following:
merged_monthly_df_clean_with_changes
#merged_yearly_df_clean_with_changes


## Preliminary Analysis (EDA)

In [ ]:
merged_monthly_df_clean_with_changes['SP500 Index Value ($)'].plot()

In [ ]:
s = merged_monthly_df_clean_with_changes['SP500 CHANGE RATE'].dropna()
s.describe(percentiles=[.01,.05,.25,.5,.75,.95,.99])

In [ ]:
cols = ['SP500 CHANGE RATE',
        'FEDFUNDS CHANGE RATE',
        'CPI CHANGE RATE',
        'UNEMPLOYMENT RATE CHANGE RATE',
        'VIX CHANGE RATE',
        'GPRC CHANGE RATE']
merged_monthly_df_clean_with_changes[cols].corr()['SP500 CHANGE RATE']

In [ ]:
# Putting unemploymenr rate and S&P500 together, we can see that their trends
# are opposite 
ax = merged_monthly_df_clean_with_changes['SP500 Index Value ($)'].plot(
    figsize=(10, 5),
    label='S&P 500',
    color='C0'
)

merged_monthly_df_clean_with_changes['UNEMPLOYMENT RATE'].plot(
    ax=ax,
    secondary_y=True,
    label='Unemployment Rate',
    color='C1'
)

ax.set_ylabel('S&P 500 Index')
ax.right_ax.set_ylabel('Unemployment (%)')

ax.legend(loc='upper left')
ax.right_ax.legend(loc='upper right')
ax.set_title('S&P 500 Index Adjusted and Unemployment Rate Since 1990')


In [ ]:
import matplotlib.dates as mdates

df = merged_monthly_df_clean_with_changes.copy()

# Ensure datetime index
df.index = pd.to_datetime(df.index)

fig, ax = plt.subplots(figsize=(12, 6))

# Left axis: S&P 500 level
sp_line = ax.plot(df.index, df['SP500 Index Value ($)'], label='S&P 500 (level)')

ax.set_ylabel('S&P 500 Index Level')
ax.set_xlabel('Date')

# Right axis: Geopolitical Risk (GPRC_USA)
ax2 = ax.twinx()
gprc_line = ax2.plot(df.index, df['GPRC_USA'], linestyle='--', label='GPRC (USA)',color='orange')
ax2.set_ylabel('Geopolitical Risk Index (GPRC_USA)')

# --- Event windows (shaded) ---
# Gulf War: Aug 1990 - Mar 1991
ax.axvspan(pd.Timestamp('1990-08-02'), pd.Timestamp('1991-2-28'), alpha=0.12, label='1990-1991 Gulf War')
#911 Panic: Sep 2001 - Jun 2003
ax.axvspan(pd.Timestamp('2001-09-11'), pd.Timestamp('2005-01-01'), alpha=0.12, label='2001 911 Attack Panic Window')
# NBER Great Recession window (approx): Dec 2007–Jun 2009
ax.axvspan(pd.Timestamp('2007-12-01'), pd.Timestamp('2009-06-30'), alpha=0.12, label='2008 Financial Crisis window')
# COVID recession (approx): Feb–Apr 2020
ax.axvspan(pd.Timestamp('2020-02-01'), pd.Timestamp('2020-04-30'), alpha=0.12, label='2020 COVID recession window')

# Gulf war
ax.axvline(pd.Timestamp('1990-08-31'),linestyle=':', linewidth=1)
ax2.annotate('Gulf War Begins (1990-8-02)', xy=(pd.Timestamp('1990-08-31'), 
             df['GPRC_USA'].loc['1990-08-31']),
            xytext=(15, 20), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', lw=0.8))
# 911 Attack
ax.axvline(pd.Timestamp('2001-09-11'),linestyle=':', linewidth=1)
ax2.annotate('9-11 Attack (2001-09-11)', xy=(pd.Timestamp('2001-09-11'), 
             df['GPRC_USA'].loc['2001-09-28']),
            xytext=(15, 20), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', lw=0.8))
# Iraq Attack
ax.axvline(pd.Timestamp('2003-03-20'),linestyle=':', linewidth=1)
ax2.annotate('Iraq War Begins', xy=(pd.Timestamp('2003-03-20'), 
             df['GPRC_USA'].loc['2003-03-31']),
            xytext=(15, 20), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', lw=0.8))
# Lehman Brothers collapse
ax.axvline(pd.Timestamp('2008-09-15'), linestyle=':', linewidth=1)
ax.annotate('Lehman collapse (2008-09-15)', xy=(pd.Timestamp('2008-09-15'), 
             df['SP500 Index Value ($)'].loc['2008-09-30']),
            xytext=(15, 20), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', lw=0.8))

# WHO declares COVID-19 pandemic
ax.axvline(pd.Timestamp('2020-03-11'), linestyle=':', linewidth=1)
# pick a y close to the series around that date to avoid None
y_2020 = df['SP500 Index Value ($)'].loc['2020-03-31']
ax.annotate('WHO pandemic declared (2020-03-11)',
            xy=(pd.Timestamp('2020-03-11'), y_2020),
            xytext=(15, -25), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', lw=0.8))
#Russia Declare War on Ukraine
ax.axvline(pd.Timestamp('2022-02-24'), linestyle=':', linewidth=1)
y_2022 = df['SP500 Index Value ($)'].loc['2022-02-28']
ax.annotate('Ukraine War (2022-02-24)',
            xy=(pd.Timestamp('2022-02-28'), y_2022),
            xytext=(15, -25), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', lw=0.8))
# Formatting: yearly ticks
ax.xaxis.set_major_locator(mdates.YearLocator(base=2))  # every 2 years
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Build one combined legend
lines = sp_line + gprc_line
labels = [l.get_label() for l in lines]
legend1 = ax.legend(lines, labels, loc='upper left')
legend2 = ax.legend(loc='upper right')  # picks up the span labels
ax.add_artist(legend1)

ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## Research Question 1: Which Major Financial Feature(s) affect S&P 500?
## 1. Stationarity Testing

Before running any regression, we test whether the variables are **stationary**, which is required for valid inference in time-series analysis.

### **Tests Applied**
- **ADF (Augmented Dickey–Fuller)**  
  - Null: series is **non-stationary**
- **KPSS (Kwiatkowski–Phillips–Schmidt–Shin)**  
  - Null: series is **stationary**
- **Ljung–Box**  
  - Tests autocorrelation in returns

### **Results**
- **S&P 500 Index Level**
  - ADF: fail to reject non-stationarity  
  - KPSS: reject stationarity  
  ➝ **Index level is non-stationary** (typical for price series)

- **S&P 500 Return (Rate of Change)**
  - ADF: reject non-stationarity  
  - KPSS: fail to reject stationarity  
  - Ljung–Box: no autocorrelation  
  ➝ **Returns are stationary and suitable for regression**

## **Final Conclusion**

- The **S&P 500 index level is non-stationary**, so it cannot be used for inference.  
- The **rate of change (return) is stationary**, making it valid for regression.  

In [ ]:
#Hypothesis Test, it's clear that S&P 500 adjusted for inflation is not
#staionary, what about the rate of change?
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
df = merged_monthly_df_clean_with_changes.copy()
s = df['SP500 Index Value ($)'].dropna()
r = df['SP500 CHANGE RATE'].dropna()

#ADF Null Hypothesis: S&P 500 Rate of Change is non-stationary
#ADF Alternative Hypothesis: S&P 500 Rate of Change is statinoary
#Result: Rejected Null
print('ADF S&P level:', adfuller(s)[1], ' | ADF S&P return:', adfuller(r)[1])
#KPSS Null Hypothesis: S&P 500 Rate of Change is stationary
#KPSS Alternative Hypothesis: S&P 500 Rate of Change is not statinoary
#Result: Fail to reject null
print('KPSS S&P level:', kpss(s, regression='c')[1], ' | KPSS S&P return:', kpss(r, regression='c')[1])

# Autocorrelation in returns
sm.stats.acorr_ljungbox(r, lags=[6,12], return_df=True)
#Result: No correlation discovered
#CONCLUSION: The results show that the rate of change is stationary, and cofirms that
#S&P500 index is non-stationary

In [ ]:
import statsmodels.api as sm
df = merged_monthly_df_clean_with_changes.copy()


# Rename columns to simpler names (adjust if your exact names differ)
df = df.rename(columns={
    "SP500 CHANGE RATE": "sp500_ret",
    "GPRC_USA": "gpr",
    "GPRC CHANGE RATE": "gpr_chg"
})

# Keep only the columns we need and drop missing values
data = df[["sp500_ret", "gpr", "gpr_chg"]].dropna()

# ===== 1. Quick exploratory plots =====
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
axes[0].plot(data.index, data["sp500_ret"])
axes[0].set_title("S&P 500 Monthly Return")

axes[1].plot(data.index, data["gpr"])
axes[1].set_title("Geopolitical Risk (GPR) Level")

axes[2].plot(data.index, data["gpr_chg"])
axes[2].set_title("Change in Geopolitical Risk (ΔGPR)")

plt.tight_layout()
plt.show()

# ===== 2. Simple correlations =====
print("Correlation matrix (returns vs GPR variables):")
print(data[["sp500_ret", "gpr", "gpr_chg"]].corr())

# ===== 3. OLS regressions with HAC (Newey–West) standard errors =====
# We use 12 lags since this is monthly data (1 year)

y = data["sp500_ret"]

# --- Model 1: Return ~ GPR level ---
X1 = sm.add_constant(data["gpr"])
model1 = sm.OLS(y, X1).fit(cov_type="HAC", cov_kwds={"maxlags": 12})
print("\n=== Model 1: sp500_ret ~ GPR (level) ===")
print(model1.summary())

# --- Model 2: Return ~ GPR change ---
X2 = sm.add_constant(data["gpr_chg"])
model2 = sm.OLS(y, X2).fit(cov_type="HAC", cov_kwds={"maxlags": 12})
print("\n=== Model 2: sp500_ret ~ ΔGPR ===")
print(model2.summary())

# --- Model 3: Return ~ GPR level + GPR change ---
X3 = sm.add_constant(data[["gpr", "gpr_chg"]])
model3 = sm.OLS(y, X3).fit(cov_type="HAC", cov_kwds={"maxlags": 12})
print("\n=== Model 3: sp500_ret ~ GPR + ΔGPR ===")
print(model3.summary())

# ===== 4. Optional: residual diagnostics for the preferred model =====
resid = model3.resid

fig, axes = plt.subplots(2, 1, figsize=(8, 6))
axes[0].plot(resid)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title("Residuals over time (Model 3)")

sm.graphics.tsa.plot_acf(resid, lags=24, ax=axes[1])
axes[1].set_title("ACF of residuals (Model 3)")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
#Compute a lead–lag Spearman correlation analysis to identify whether macro/geopolitical variables lead (predict) S&P 500 monthly returns at lags 0–6.
merged_df = merged_monthly_df_clean_with_changes.copy()

# Target: month-to-month change in S&P 500
sp = merged_df['SP500 CHANGE RATE']

# Predictors: month-to-month changes in the other indicators
predictors = [
    'GPRC CHANGE RATE',
    'FEDFUNDS CHANGE RATE',
    'CPI CHANGE RATE',
    'UNEMPLOYMENT RATE CHANGE RATE',
    'VIX CHANGE RATE'
]

lag_results = []

for var in predictors:
    for lag in range(0, 7):  # 0 = same month, 1–6 months lag
        shifted = merged_df[var].shift(lag)
        corr = sp.corr(shifted, method='spearman')  
        lag_results.append([var, lag, corr])

lag_df = pd.DataFrame(
    lag_results,
    columns=['Variable', 'Lag_months', 'Spearman_Correlation']
).sort_values(by='Spearman_Correlation', ascending=False)

#return variables with Spearman correlation above a threshold of 0.20
lag_df[lag_df['Spearman_Correlation'].abs() > 0.20]


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
# Compute rolling 6-month cumulative geopolitical pressure and rolling 6-month S&P volatility
# to test whether prolonged geopolitical tension corresponds to prolonged market instability.

window = 6  # 6 months

# 6-month cumulative geopolitical pressure
merged_df['GPR_pressure_6m'] = merged_df['GPRC CHANGE RATE'].rolling(window).sum()

# 6-month rolling volatility of S&P 500 returns
merged_df['SP500_vol_6m'] = merged_df['SP500 CHANGE RATE'].rolling(window).std()

merged_df[['GPR_pressure_6m', 'SP500_vol_6m']].tail(10)

In [ ]:
merged_df[['GPR_pressure_6m', 'SP500_vol_6m']].corr(method='spearman')

In [ ]:
from scipy.stats import f_oneway
#Perform anova tests with SP500 monthly returns as the dependent variable, across low, medium, and high regimes
# for each economic/geopolitical indicator(based on quantile bins). 
def anova_test(df, group_col, ret_col="SP500 CHANGE RATE"):
    df = df.copy()
    #Test based on a quantile cut
    df["group"] = pd.qcut(df[group_col], 3, labels=["Low", "Med", "High"])
    g1 = df[df["group"]=="Low"][ret_col]
    g2 = df[df["group"]=="Med"][ret_col]
    g3 = df[df["group"]=="High"][ret_col]
    F, p = f_oneway(g1, g2, g3)
    return {"Variable": group_col, "F": F, "p": p}

variables = [
    "GPRC CHANGE RATE", "VIX CHANGE RATE", "CPI CHANGE RATE",
    "FEDFUNDS CHANGE RATE", "UNEMPLOYMENT RATE CHANGE RATE"
]

anova_results = [anova_test(merged_df, var) for var in variables]
pd.DataFrame(anova_results).sort_values("p")


## 📌 Summary of Analysis

Our analysis examined whether geopolitical risk and major macroeconomic indicators (GPR, Federal Funds Rate, CPI, Unemployment, VIX) meaningfully influence monthly S&P 500 returns. We applied multiple statistical approaches to evaluate both predictive power and regime effects.

---

### 🔷 **1. Lead–Lag Correlation (0–6 Months)**

We tested whether monthly macro and geopolitical indicators *lead* S&P 500 returns by 0–6 months using Spearman correlation.

> **Result:** Correlations were consistently very small (**|ρ| < 0.10** across all variables and lags)

📌 **Interpretation:** These variables do **not** meaningfully predict S&P 500 returns over the following months.  
Markets appear to price in macro/geopolitical information quickly.

---

### 🔷 **2. Rolling 6-Month Window Analysis**

We constructed:
- **6-month cumulative geopolitical pressure**, and
- **6-month rolling volatility of S&P 500 returns**

> **Result:** Spearman correlation ≈ **0.106** between rolling GPR pressure and S&P 500 volatility

📌 **Interpretation:** Sustained geopolitical tension is associated with **higher market volatility**,  
but **does not reliably predict whether returns will be positive or negative**.

---

### 🔷 **3. ANOVA Regime Testing**

Each indicator was split into **Low / Medium / High** regimes (via quantile bins), and we tested whether mean S&P 500 returns differed across the regimes.

| Indicator | Statistically Significant? |
|----------|-----------------------------|
| **VIX Change Rate** | ✔ **Yes (p ≪ 0.001)** |
| GPR Change Rate | ❌ No |
| Federal Funds Change Rate | ❌ No |
| CPI Change Rate | ❌ No |
| Unemployment Rate Change Rate | ❌ No |

📌 **Interpretation:** Only **rapid fluctuations in volatility (VIX)** correspond to meaningful differences in market return environments. Other indicators do not shift average return outcomes.

---

### 🔷 **4. Inference: Geopolitical Risk → S&P 500 Returns**

We regress monthly S&P 500 returns on:

- **GPR level** (GPR)  
- **Change in GPR** (ΔGPR)  

using **HAC (Newey–West)** robust standard errors.

### **Key Findings**

#### **Model 1 — Return ~ GPR Level**
- Coefficient not significant  
- No explanatory power  
➡️ **Background geopolitical tension does not affect returns**

#### **Model 2 — Return ~ ΔGPR**
- Coefficient **negative and highly significant**  
- Returns fall when geopolitical risk increases  
➡️ **Markets respond to geopolitical shocks, not levels**

#### **Model 3 — Return ~ GPR + ΔGPR**
- GPR level remains insignificant  
- ΔGPR remains significantly negative  
➡️ **Effect of geopolitical shocks is robust**

**In short:**  
> The S&P 500 reacts to **changes** in geopolitical risk, not to the steady level of geopolitical tension.


## Research Question 2
## Is GPR Collinear With Other Financial Features

In [ ]:
df.columns

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# 1. Select the correct merged dataframe
# ============================================================

if 'merged_monthly_df_clean_with_changes' in globals():
    df = merged_monthly_df_clean_with_changes.copy()
elif 'merged_monthly_df_with_changes' in globals():
    df = merged_monthly_df_with_changes.copy()
else:
    df = merged_monthly_df.copy()

# ============================================================
# 2. Choose correct column names for collinearity analysis
# ============================================================

cols = [
    "GPRC_USA",
    "FEDFUNDS",
    "CPI",
    "UNEMPLOYMENT RATE",
    "VIX"
]

# Drop missing rows to compute correlations cleanly
df_corr = df[cols].dropna()

# ============================================================
# 3. Compute Pearson correlation matrix
# ============================================================

corr_matrix = df_corr.corr(method="pearson")
corr_matrix


In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Pairwise Correlation Matrix — GPR vs Financial Indicators")
plt.show()

## Collinearity Test: Does GPR or GPR Change Overlap With Financial Indicators?

### **Rationale**
To ensure our regression results are trustworthy, we must determine whether Geopolitical Risk (GPR) or GPR Change is *collinear* with core financial indicators (VIX, Fed Funds, CPI, unemployment).  
If GPR is redundant with any of these variables, it could explain why GPR appears statistically weak in predicting S&P 500 returns.

### **Method**
Following the coefficient-stability method from *Introduction to Modern Statistics* (IMS Ch. 25), we test for collinearity by examining whether:

1. Regression coefficients change sign,
2. Magnitudes swing sharply,
3. Significance disappears or appears unexpectedly,
4. Coefficients for GPR or GPR Change behave erratically when both are included.

For each financial variable $X$, we estimate:
- Model 1: $X \sim GPR$
- Model 2: $X \sim \Delta\text{GPR}$
- Model 3: $X \sim \text{GPR} + \Delta\text{GPR}$

### **Results**

- **Fed Funds, CPI, Unemployment:**  
  Coefficients for GPR and GPR Change are small, stable, and mostly insignificant.  
  → **No collinearity.**

- **VIX (Level):**  
  Small coefficient shrinkage but same sign and stable significance.  
  → **Shared uncertainty signal, not collinearity.**

- **VIX Change Rate:**  
  GPR Change strongly predicts VIX Change (p < 0.001).  
  GPR flips sign when GPR Change is included.  
  → Indicates **shared explanatory signal**, but coefficients remain stable and interpretable.  
  → **Not harmful multicollinearity.**

### **Conclusion**
Neither GPR nor GPR Change is collinear with any of the major financial indicators.  
The only partial overlap occurs with **VIX Change Rate**, where both variables capture market uncertainty shocks, but this does not introduce harmful multicollinearity.

Therefore, the weak predictive effect of GPR on S&P 500 returns cannot be explained by collinearity with macroeconomic or volatility variables.


In [ ]:
features = [
    'FEDFUNDS', 'FEDFUNDS CHANGE RATE',
    'CPI', 'CPI CHANGE RATE',
    'UNEMPLOYMENT RATE', 'UNEMPLOYMENT RATE CHANGE RATE',
    'VIX', 'VIX CHANGE RATE'
]
df = merged_monthly_df_clean_with_changes.copy()
results = []

def run_regression(y, Xvars):
    X = df[Xvars]
    X = sm.add_constant(X)
    model = sm.OLS(df[y], X, missing='drop').fit()
    return model


for feat in features:

    # Model A: feature ~ GPR
    m1 = run_regression(feat, ['GPRC_USA'])

    # Model B: feature ~ GPR change
    m2 = run_regression(feat, ['GPRC CHANGE RATE'])

    # Model C: feature ~ GPR + GPR change
    m3 = run_regression(feat, ['GPRC_USA', 'GPRC CHANGE RATE'])

    results.append({
        'feature': feat,
        'coef_GPR_m1': m1.params.get('GPRC_USA', None),
        'coef_GPRchg_m2': m2.params.get('GPRC CHANGE RATE', None),
        'coef_GPR_m3': m3.params.get('GPRC_USA', None),
        'coef_GPRchg_m3': m3.params.get('GPRC CHANGE RATE', None),
        'p_GPR_m1': m1.pvalues.get('GPRC_USA', None),
        'p_GPRchg_m2': m2.pvalues.get('GPRC CHANGE RATE', None),
        'p_GPR_m3': m3.pvalues.get('GPRC_USA', None),
        'p_GPRchg_m3': m3.pvalues.get('GPRC CHANGE RATE', None),
    })
collinearity_df = pd.DataFrame(results)
collinearity_df


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Pick dataframe
if 'merged_monthly_df_with_changes' in globals():
    df = merged_monthly_df_with_changes.copy()
elif 'merged_monthly_df' in globals():
    df = merged_monthly_df.copy()
else:
    raise ValueError("Run the earlier cells that create the monthly merged dataframe first.")

# 2. Ensure datetime index
if not isinstance(df.index, pd.DatetimeIndex):
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.set_index('Date')
    elif 'date_month' in df.columns:
        df['date_month'] = pd.to_datetime(df['date_month'].astype(str).str.replace('.', '-') + '-01', errors='coerce')
        df = df.set_index('date_month')
    else:
        raise ValueError("No 'Date' or 'date_month' column available.")

df = df.sort_index()

# 3. Auto-detect columns
def find_col(keyword):
    keyword = keyword.upper()
    candidates = [c for c in df.columns if keyword in c.upper()]
    return candidates[0] if candidates else None

sp_col = find_col('SP500')
gpr_col = find_col('GPR')
cpi_col = find_col('CPI')
unemp_col = find_col('UNEMPLOYMENT')
vix_col = find_col('VIX')

cols_to_plot = [c for c in [sp_col, gpr_col, cpi_col, unemp_col, vix_col] if c is not None]

if len(cols_to_plot) < 2:
    raise ValueError("Not enough expected columns found (SP500, GPR, CPI, UNEMPLOYMENT, VIX).")

plot_df = df[cols_to_plot].dropna()

# 4. Normalize each series
norm_df = plot_df.copy()
for col in norm_df.columns:
    col_min = norm_df[col].min()
    col_max = norm_df[col].max()
    if col_max > col_min:
        norm_df[col] = (norm_df[col] - col_min) / (col_max - col_min)

# 5. Plot
plt.figure(figsize=(12, 6))
for col in norm_df.columns:
    plt.plot(norm_df.index, norm_df[col], label=col, linewidth=2)

plt.title("Normalized Monthly Trends in S&P 500 and Key Variables")
plt.xlabel("Year")
plt.ylabel("Normalized Value (0–1)")
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()